# Lab 11: 1D Convolutional Neural Network (CNN) for Time Series Forecasting

## Objective
Build and train a 1D Convolutional Neural Network to forecast the next time step of a multivariate time series dataset (energy consumption). The model uses a **univariate multi‑step** input – it sees a window of past values (24 hours) and predicts the next value of the target variable.

## Dataset
- The dataset contains hourly energy consumption data (AEP).
- Features include temporal indicators (hour, day of week, month, etc.) and lagged values.
- Data is already split into `train.csv`, `validation.csv`, and `test.csv`, and pre‑scaled using a `MinMaxScaler`.

## Model Architecture (1D CNN)
The CNN consists of:
- **Conv1D** layer with 16 filters, kernel size 2, ReLU activation.
- **Conv1D** layer with 16 filters, kernel size 2, ReLU activation.
- **Flatten** layer to convert the 2D feature maps into a 1D vector.
- **Dense** output layer with 1 unit (linear activation) for regression.

## Training
- **Loss:** Mean Absolute Error (MAE)
- **Optimizer:** Adam with learning rate 1e‑3 (then fine‑tuned with 1e‑4)
- **Metrics:** MAE and MAPE
- **Callbacks:** ModelCheckpoint (save best) and TrainingMonitor (plot history)

## Fine‑Tuning
After the initial training, we load the best saved model, reduce the learning rate, and continue training for a few more epochs to further improve performance.

## Evaluation
The final model is evaluated on the test set using several error metrics (MAE, MedAE, MSE, RMSE, MAPE, MDAPE).

---

In [1]:
# 1. SETUP AND IMPORTS

import os
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# TensorFlow / Keras
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import (
    Dense, Dropout, Activation, Flatten, Input, Reshape, Lambda,
    Concatenate, Conv1D, MaxPooling1D, AveragePooling1D,
    GlobalMaxPooling1D, BatchNormalization, TimeDistributed,
    LSTM, Bidirectional, Add, Layer, LeakyReLU
)
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
from tensorflow.keras.regularizers import l2

# Custom utility modules (from the `timeseires` package)
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor

# Scikit‑learn metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score

In [2]:
# 2. PATHS AND PARAMETERS

# Change working directory to dataset location
os.chdir(r'C:\Users\PMYLS\Documents\MachineLearningLaB\Dataset')

# Global parameters
time_steps = 24          # lookback window (24 hours)
num_features = 21        # number of input features
target_col = 0           # column index of the target variable

# Paths for saved models and outputs
OUTPUT_PATH = r'C:\Users\PMYLS\Documents\MachineLearningLaB\Dataset'
CHECKPOINT_PATTERN = os.path.join(OUTPUT_PATH, 'cnn-{epoch:04d}-loss{val_loss:.2f}.h5')
FIG_PATH = os.path.join(OUTPUT_PATH, 'cnn_history.png')
JSON_PATH = os.path.join(OUTPUT_PATH, 'cnn_history.json')

## 3. Define the 1D CNN Model

In [3]:
def build_cnn():
    """Create a 1D CNN model for time series regression."""
    input_data = Input(shape=(time_steps, num_features))
    x = Conv1D(16, kernel_size=2, activation='relu')(input_data)
    x = Conv1D(16, kernel_size=2, activation='relu')(x)
    x = Flatten()(x)
    output_data = Dense(1)(x)   # linear output for regression
    model = Model(inputs=input_data, outputs=output_data)
    return model

In [4]:
# Instantiate and display the model
model = build_cnn()
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 24, 21)]          0         
                                                                 
 conv1d (Conv1D)             (None, 23, 16)            688       
                                                                 
 conv1d_1 (Conv1D)           (None, 22, 16)            528       
                                                                 
 flatten (Flatten)           (None, 352)               0         
                                                                 
 dense (Dense)               (None, 1)                 353       
                                                                 
Total params: 1,569
Trainable params: 1,569
Non-trainable params: 0
_________________________________________________________________


## 4. Compile the Model

In [5]:
# If no pre‑trained model is given, compile from scratch
if model is None:
    print("[INFO] compiling model...")
    model = build_cnn()
    opt = Adam(learning_rate=1e-3)
    model.compile(loss='mae', optimizer=opt, metrics=['mae', 'mape'])
else:
    print("[INFO] loading existing model...")
    model = load_model(model)
    # Optionally adjust learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


## 5. Load and Prepare the Data

The data is already scaled and split into train/validation/test sets. We also load the scaler to invert predictions later.

In [6]:
# Read CSV files
train_df = pd.read_csv('train.csv')
validation_df = pd.read_csv('validation.csv')
test_df = pd.read_csv('test.csv')

# Convert to numpy arrays
train_set = train_df.values
validation_set = validation_df.values
test_set = test_df.values

# Load the scaler
scaler = pickle.load(open('AEP_scaler.pkl', 'rb'))

train_set.shape, validation_set.shape, test_set.shape

C:\Users\PMYLS\anaconda3\envs\machinelearning\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

## 6. Create Sequences (Univariate Multi‑Step)

We convert the time series into overlapping windows of length `time_steps` (24 hours) to predict the next single value (target_len=1).

In [7]:
start = time.time()
train_X, train_y = univariate_multi_step(train_set, time_steps, target_col=target_col, target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=target_col, target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=target_col, target_len=1)
print('Time Consumed', time.time()-start, "sec")

# Show shapes
print(f"train_X shape: {train_X.shape}, train_y shape: {train_y.shape}")
print(f"validation_X shape: {validation_X.shape}, validation_y shape: {validation_y.shape}")
print(f"test_X shape: {test_X.shape}, test_y shape: {test_y.shape}")

Time Consumed 0.015037298202514648 sec


## 7. Set Up Callbacks

- `ModelCheckpoint` saves the model with the best validation loss.
- `TrainingMonitor` logs training history and plots it.

In [8]:
checkpoint = ModelCheckpoint(
    CHECKPOINT_PATTERN,
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)
monitor = TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=0)
callbacks = [checkpoint, monitor]

## 8. Train the Model (Initial Run)

In [9]:
epochs = 9
batch_size = 32

history = model.fit(
    train_X, train_y,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(validation_X, validation_y),
    callbacks=callbacks,
    verbose=1
)

Epoch 1/9
27/27 [==============================] - ETA: 0s - loss: 0.1361 - mae: 0.1361 - mape: 60.4483
Epoch 1: val_loss improved from inf to 0.06399, saving model to C:\Users\PMYLS\Documents\MachineLearningLaB\Dataset\cnn-0001-loss0.06.h5
27/27 [==============================] - 4s 68ms/step - loss: 0.1361 - mae: 0.1361 - mape: 60.4483 - val_loss: 0.0640 - val_mae: 0.0640 - val_mape: 23.9688
Epoch 2/9
27/27 [==============================] - ETA: 0s - loss: 0.0648 - mae: 0.0648 - mape: 30.1789
Epoch 2: val_loss improved from 0.06399 to 0.03769, saving model to C:\Users\PMYLS\Documents\MachineLearningLaB\Dataset\cnn-0002-loss0.04.h5
27/27 [==============================] - 1s 42ms/step - loss: 0.0648 - mae: 0.0648 - mape: 30.1789 - val_loss: 0.0377 - val_mae: 0.0377 - val_mape: 12.9567
Epoch 3/9
22/27 [=======================>......] - ETA: 0s - loss: 0.0545 - mae: 0.0545 - mape: 30.8423
Epoch 3: val_loss did not improve from 0.03769
27/27 [==============================] - 1s 37ms/st

## 9. Evaluate the Initial Model

We load the best saved model (based on validation loss) and compute error metrics on the test set.

In [10]:
# Load the best model from the initial training
best_model_path = os.path.join(OUTPUT_PATH, 'cnn-0002-loss0.04.h5')
model_best = load_model(best_model_path)

# Predict on test set
y_pred_scaled = model_best.predict(test_X)
y_pred = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)

# Compute metrics
MAE = np.mean(abs(y_pred - y_test_unscaled))
MEDAE = np.median(abs(y_pred - y_test_unscaled))
MSE = np.mean((y_pred - y_test_unscaled)**2)
RMSE = np.sqrt(MSE)
MAPE = np.mean(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100
MDAPE = np.median(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100

print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')
print('\n\ny_test_unscaled.shape= ', y_test_unscaled.shape)
print('y_pred.shape= ', y_pred.shape)

1/1 [==============================] - 0s 223ms/step
Mean Absolute Error (MAE): 2989.68
Median Absolute Error (MedAE): 2872.05
Mean Squared Error (MSE): 9860633.10
Root Mean Squared Error (RMSE): 3140.16
Mean Absolute Percentage Error (MAPE): 19.0 %
Median Absolute Percentage Error (MDAPE): 18.56 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## 10. Fine‑Tuning the Model

We continue training from the best checkpoint with a lower learning rate to further improve performance.

In [11]:
# Path to the model to fine‑tune
finetune_model_path = os.path.join(OUTPUT_PATH, 'cnn-0002-loss0.04.h5')

# Load the model
model_ft = load_model(finetune_model_path)

# Reduce learning rate for fine‑tuning
K.set_value(model_ft.optimizer.lr, 1e-4)
print("[INFO] new learning rate: {}".format(K.get_value(model_ft.optimizer.lr)))

# Re‑compile (optional but safe)
model_ft.compile(loss='mae', optimizer=model_ft.optimizer, metrics=['mae', 'mape'])

# Set up callbacks – use a distinct checkpoint pattern for fine‑tuning
ft_checkpoint_path = os.path.join(OUTPUT_PATH, 'cnn-ft-{epoch:04d}-loss{val_loss:.2f}.h5')
ft_checkpoint = ModelCheckpoint(ft_checkpoint_path, monitor='val_loss', save_best_only=True, verbose=1)
ft_monitor = TrainingMonitor(FIG_PATH.replace('.png', '_ft.png'), jsonPath=JSON_PATH.replace('.json', '_ft.json'), startAt=0)
ft_callbacks = [ft_checkpoint, ft_monitor]

[INFO] loading C:\Users\PMYLS\Documents\MachineLearningLaB\Dataset\cnn-0002-loss0.04.h5...
[INFO] old learning rate: 0.0010000000474974513
[INFO] new learning rate: 9.999999747378752e-05


In [12]:
# Fine‑tune for additional epochs
ft_epochs = 10
history_ft = model_ft.fit(
    train_X, train_y,
    batch_size=batch_size,
    epochs=ft_epochs,
    validation_data=(validation_X, validation_y),
    callbacks=ft_callbacks,
    verbose=1
)

Epoch 1/10
27/27 [==============================] - ETA: 0s - loss: 0.0541 - mae: 0.0541 - mape: 28.0265
Epoch 1: val_loss improved from inf to 0.03725, saving model to C:\Users\PMYLS\Documents\MachineLearningLaB\Dataset\cnn-ft-0001-loss0.04.h5
27/27 [==============================] - 2s 40ms/step - loss: 0.0541 - mae: 0.0541 - mape: 28.0265 - val_loss: 0.0373 - val_mae: 0.0373 - val_mape: 12.7451
Epoch 2/10
20/27 [=====================>........] - ETA: 0s - loss: 0.0525 - mae: 0.0525 - mape: 28.4792
Epoch 2: val_loss did not improve from 0.03725
27/27 [==============================] - 1s 23ms/step - loss: 0.0520 - mae: 0.0520 - mape: 27.1432 - val_loss: 0.0373 - val_mae: 0.0373 - val_mape: 12.6336
Epoch 3/10
25/27 [==========================>...] - ETA: 0s - loss: 0.0513 - mae: 0.0513 - mape: 26.2300
Epoch 3: val_loss did not improve from 0.03725
27/27 [==============================] - 1s 24ms/step - loss: 0.0509 - mae: 0.0509 - mape: 26.6466 - val_loss: 0.0400 - val_mae: 0.0400 - v

## 11. Evaluate the Fine‑Tuned Model

In [13]:
# Load the best fine‑tuned model
best_ft_path = os.path.join(OUTPUT_PATH, 'cnn-ft-0001-loss0.04.h5')
model_ft_best = load_model(best_ft_path)

# Predict and evaluate
y_pred_ft_scaled = model_ft_best.predict(test_X)
y_pred_ft = scaler.inverse_transform(y_pred_ft_scaled)

# Compute metrics
MAE_ft = np.mean(abs(y_pred_ft - y_test_unscaled))
MEDAE_ft = np.median(abs(y_pred_ft - y_test_unscaled))
MSE_ft = np.mean((y_pred_ft - y_test_unscaled)**2)
RMSE_ft = np.sqrt(MSE_ft)
MAPE_ft = np.mean(np.abs((y_test_unscaled - y_pred_ft) / y_test_unscaled)) * 100
MDAPE_ft = np.median(np.abs((y_test_unscaled - y_pred_ft) / y_test_unscaled)) * 100

print('Mean Absolute Error (MAE): ' + str(np.round(MAE_ft, 2)))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE_ft, 2)))
print('Mean Squared Error (MSE): ' + str(np.round(MSE_ft, 2)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE_ft, 2)))
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE_ft, 2)) + ' %')
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE_ft, 2)) + ' %')
print('\n\ny_test_unscaled.shape= ', y_test_unscaled.shape)
print('y_pred_ft.shape= ', y_pred_ft.shape)

1/1 [==============================] - 0s 207ms/step
Mean Absolute Error (MAE): 2800.63
Median Absolute Error (MedAE): 2691.45
Mean Squared Error (MSE): 8740589.14
Root Mean Squared Error (RMSE): 2956.45
Mean Absolute Percentage Error (MAPE): 17.8 %
Median Absolute Percentage Error (MDAPE): 17.39 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## 12. Summary and Comparison

We compare the initial and fine‑tuned model performance on the test set.

| Metric | Initial | Fine‑Tuned | Improvement |
|--------|---------|------------|-------------|
| MAE    | 2989.68 | 2800.63    | -6.32%      |
| RMSE   | 3140.16 | 2956.45    | -5.85%      |
| MAPE   | 19.0%   | 17.8%      | -1.2 pp     |

Fine‑tuning with a lower learning rate improved all metrics, demonstrating the benefit of continued training with a smaller step size. The 1D CNN model performs well on this time series forecasting task, capturing local patterns in the 24‑hour input window.